In [1]:
#!/usr/bin/env python3
"""
DsRed: classificacao por bright pixels (p85) + Otsu + overlays com GT real (XM/YM do Fiji).
"""

import os
import numpy as np
import pandas as pd
import tifffile as tiff
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from pathlib import Path

from skimage.filters import threshold_otsu

# -----------------------------
# CONFIG
# -----------------------------
RESULTS_DIR = r"C:\Users\OSVALDO\Downloads\results"
OUT_DIR = r"C:\Users\OSVALDO\Downloads\results\03fev\overlays_realGT_p85_otsu"
os.makedirs(OUT_DIR, exist_ok=True)

SLICES = [110, 300, 450]
GT_CSV_BY_SLICE = {
    110: r"C:\Users\OSVALDO\Downloads\results\Results110dsrednovo.csv",
    300: r"C:\Users\OSVALDO\Downloads\results\Results300dsrednovo.csv",
    450: r"C:\Users\OSVALDO\Downloads\results\Results450dsrednovo.csv",
}

# Bright pixels
BRIGHT_PERCENTILE = 85.0  # global no slice
NEIGHBOR_RADIUS_PX = 3    # para mapear pontos GT a labels quando caem fora da mascara
MIN_CELL_PIXELS = 20      # ignora labels muito pequenos

# -----------------------------
# IO
# -----------------------------
def load_tiff_channels(tiff_path):
    img = tiff.imread(tiff_path)
    if img.ndim == 3 and img.shape[0] == 2:       # (2, H, W)
        dsred = img[0]
        mask = img[1]
    elif img.ndim == 3 and img.shape[-1] == 2:    # (H, W, 2)
        dsred = img[..., 0]
        mask = img[..., 1]
    else:
        raise ValueError(f"Formato inesperado: {img.shape} em {tiff_path}")
    return dsred.astype(np.float32), mask.astype(np.int32)


def find_tiff_for_slice(results_dir, slice_num):
    results_path = Path(results_dir)
    patterns = [
        f"*slice{slice_num}*fish7*.tif",
        f"*slice_{slice_num}*.tif",
        f"*{slice_num}*.tif",
    ]
    for pattern in patterns:
        files = list(results_path.glob(pattern))
        if files:
            return str(files[0])
    raise FileNotFoundError(f"Nenhum TIFF encontrado para slice {slice_num} em {results_dir}")

# -----------------------------
# GT (Fiji XM/YM -> labels)
# -----------------------------
def _maybe_fix_1_based(coords, W, H):
    x = coords[:, 0]
    y = coords[:, 1]
    out0 = np.mean((x < 0) | (x >= W) | (y < 0) | (y >= H))

    x1 = x - 1
    y1 = y - 1
    out1 = np.mean((x1 < 0) | (x1 >= W) | (y1 < 0) | (y1 >= H))

    if out1 + 1e-6 < out0:
        return np.stack([x1, y1], axis=1), True
    return coords, False


def _label_at_or_near(mask, x, y, r=3):
    H, W = mask.shape
    xi = int(round(float(x)))
    yi = int(round(float(y)))

    if xi < 0 or xi >= W or yi < 0 or yi >= H:
        return 0

    lab = int(mask[yi, xi])
    if lab != 0:
        return lab

    if r <= 0:
        return 0

    x0 = max(0, xi - r)
    x1 = min(W - 1, xi + r)
    y0 = max(0, yi - r)
    y1 = min(H - 1, yi + r)

    patch = mask[y0 : y1 + 1, x0 : x1 + 1]
    vals = patch[patch > 0]
    if vals.size == 0:
        return 0

    uniq, cnt = np.unique(vals, return_counts=True)
    return int(uniq[np.argmax(cnt)])


def load_gt_positive_labels_from_fiji_csv(gt_csv_path, mask_img, neighbor_radius_px=3):
    df = pd.read_csv(gt_csv_path)
    if "XM" not in df.columns or "YM" not in df.columns:
        raise ValueError(f"CSV GT {gt_csv_path} nao tem XM/YM. Colunas: {list(df.columns)}")

    coords = df[["XM", "YM"]].to_numpy(dtype=float)
    H, W = mask_img.shape
    coords, used_minus_one = _maybe_fix_1_based(coords, W, H)

    labels = []
    n_oob = 0
    n_zero = 0
    for (x, y) in coords:
        lab = _label_at_or_near(mask_img, x, y, r=int(neighbor_radius_px))
        if lab == 0:
            xi = int(round(float(x)))
            yi = int(round(float(y)))
            if xi < 0 or xi >= W or yi < 0 or yi >= H:
                n_oob += 1
            else:
                n_zero += 1
        else:
            labels.append(lab)

    gt_labels = set(int(v) for v in labels)
    info = {
        "n_points": int(coords.shape[0]),
        "n_gt_labels_unique": int(len(gt_labels)),
        "used_minus_one_correction": bool(used_minus_one),
        "n_points_out_of_bounds": int(n_oob),
        "n_points_no_label_found": int(n_zero),
    }
    return gt_labels, info

# -----------------------------
# CLASSIFICACAO (p85 + Otsu)
# -----------------------------
def classify_cells_p85_otsu(dsred_img, mask_img, bright_percentile=85.0, min_cell_pixels=20):
    ds = dsred_img.astype(np.float32)
    m = mask_img.astype(np.int32)

    labels = np.unique(m)
    labels = labels[labels != 0]

    # limiar global de bright pixels
    ds_pos = ds[ds > 0]
    if ds_pos.size == 0:
        ds_pos = ds.reshape(-1)
    bright_thr = float(np.percentile(ds_pos, float(bright_percentile)))

    rows = []
    for lab in labels:
        pix = ds[m == int(lab)]
        if pix.size < int(min_cell_pixels):
            continue
        pct_bright = 100.0 * float(np.mean(pix > bright_thr))
        rows.append({
            "label": int(lab),
            "n_pixels": int(pix.size),
            "mean": float(np.mean(pix)),
            "median": float(np.median(pix)),
            "p85_cell": float(np.percentile(pix, 85.0)),
            "pct_bright": float(pct_bright),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("Nenhuma celula valida (df vazio). Ajuste MIN_CELL_PIXELS ou verifique a mascara.")

    # Otsu no pct_bright (entre celulas)
    vals = df["pct_bright"].to_numpy(dtype=np.float32)
    otsu_thr = float(threshold_otsu(vals))
    df["otsu_threshold_pct_bright"] = otsu_thr
    df["is_positive"] = df["pct_bright"] >= otsu_thr

    return df, {"bright_thr_intensity": bright_thr, "otsu_thr_pct_bright": otsu_thr}

# -----------------------------
# METRICAS + OVERLAYS
# -----------------------------
def categorize_by_real_gt(class_df, gt_labels):
    df = class_df.copy()
    df["gt_positive"] = df["label"].astype(int).isin(gt_labels)

    df["category"] = "TN"
    df.loc[df["is_positive"] & df["gt_positive"], "category"] = "TP"
    df.loc[df["is_positive"] & (~df["gt_positive"]), "category"] = "FP"
    df.loc[(~df["is_positive"]) & df["gt_positive"], "category"] = "FN"
    return df


def create_colored_overlay(mask_img, categorized_df):
    H, W = mask_img.shape
    overlay = np.zeros((H, W, 3), dtype=np.uint8)

    color_map = {"TP":[0,255,0], "FP":[255,0,0], "FN":[0,0,255], "TN":[100,100,100]}
    lab2cat = dict(zip(categorized_df["label"].astype(int), categorized_df["category"].astype(str)))

    for lab in np.unique(mask_img):
        if lab == 0:
            continue
        cat = lab2cat.get(int(lab), "TN")
        overlay[mask_img == int(lab)] = color_map[cat]

    return overlay


def create_visualization(dsred_img, mask_img, categorized_df, slice_num):
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    ds = dsred_img.astype(np.float32)
    ds_norm = (ds - ds.min()) / (ds.max() - ds.min() + 1e-8)

    overlay = create_colored_overlay(mask_img, categorized_df)

    counts = categorized_df["category"].value_counts()
    tp = int(counts.get("TP", 0))
    fp = int(counts.get("FP", 0))
    fn = int(counts.get("FN", 0))
    tn = int(counts.get("TN", 0))

    axes[0,0].imshow(ds_norm, cmap="hot"); axes[0,0].set_title(f"Slice {slice_num} - DsRed"); axes[0,0].axis("off")
    axes[0,1].imshow(mask_img, cmap="nipy_spectral"); axes[0,1].set_title("Mascaras"); axes[0,1].axis("off")
    axes[0,2].imshow(overlay); axes[0,2].set_title("Classificacao (GT real)"); axes[0,2].axis("off")

    def only_cat(cat, color):
        out = np.zeros_like(overlay)
        labs = categorized_df[categorized_df["category"] == cat]["label"].astype(int).values
        for lab in labs:
            out[mask_img == lab] = color
        return out

    fp_overlay = only_cat("FP", [255,0,0])
    tp_overlay = only_cat("TP", [0,255,0])
    fn_overlay = only_cat("FN", [0,0,255])

    axes[1,0].imshow(ds_norm, cmap="gray", alpha=0.5); axes[1,0].imshow(fp_overlay, alpha=0.7)
    axes[1,0].set_title(f"FALSOS POSITIVOS (FP={fp})", color="red"); axes[1,0].axis("off")

    axes[1,1].imshow(ds_norm, cmap="gray", alpha=0.5); axes[1,1].imshow(tp_overlay, alpha=0.7)
    axes[1,1].set_title(f"True Positives (TP={tp})", color="green"); axes[1,1].axis("off")

    axes[1,2].imshow(ds_norm, cmap="gray", alpha=0.5); axes[1,2].imshow(fn_overlay, alpha=0.7)
    axes[1,2].set_title(f"False Negatives (FN={fn})", color="blue"); axes[1,2].axis("off")

    legend_elements = [
        Rectangle((0,0),1,1,fc="green",label=f"TP:{tp}"),
        Rectangle((0,0),1,1,fc="red",label=f"FP:{fp}"),
        Rectangle((0,0),1,1,fc="blue",label=f"FN:{fn}"),
        Rectangle((0,0),1,1,fc="gray",label=f"TN:{tn}"),
    ]
    fig.legend(handles=legend_elements, loc="lower center", ncol=4, frameon=True)
    plt.tight_layout(rect=[0, 0.03, 1, 0.98])

    return fig, fp_overlay, {"TP":tp,"FP":fp,"FN":fn,"TN":tn}


def main():
    for slice_num in SLICES:
        print(f"\n--- Slice {slice_num} ---")
        tiff_path = find_tiff_for_slice(RESULTS_DIR, slice_num)
        dsred_img, mask_img = load_tiff_channels(tiff_path)

        gt_csv = GT_CSV_BY_SLICE[slice_num]
        gt_labels, gt_info = load_gt_positive_labels_from_fiji_csv(gt_csv, mask_img, NEIGHBOR_RADIUS_PX)
        print(f"GT points={gt_info['n_points']} -> labels_unique={gt_info['n_gt_labels_unique']} "
              f"(oob={gt_info['n_points_out_of_bounds']}, no_label={gt_info['n_points_no_label_found']})")

        class_df, thr = classify_cells_p85_otsu(
            dsred_img, mask_img,
            bright_percentile=BRIGHT_PERCENTILE,
            min_cell_pixels=MIN_CELL_PIXELS
        )
        print(f"bright_thr_intensity(p{BRIGHT_PERCENTILE})={thr['bright_thr_intensity']:.3f} | "
              f"otsu_thr_pct_bright={thr['otsu_thr_pct_bright']:.3f}")

        categorized_df = categorize_by_real_gt(class_df, gt_labels)
        fig, fp_overlay, cm = create_visualization(dsred_img, mask_img, categorized_df, slice_num)
        print(f"Confusion: TP={cm['TP']} FP={cm['FP']} FN={cm['FN']} TN={cm['TN']}")

        out_png = os.path.join(OUT_DIR, f"slice_{slice_num}_overlay_p85_otsu_realGT.png")
        fig.savefig(out_png, dpi=300, bbox_inches="tight")
        plt.close(fig)

        out_fp = os.path.join(OUT_DIR, f"slice_{slice_num}_FP_overlay_p85_otsu_realGT.tif")
        tiff.imwrite(out_fp, fp_overlay)

        out_csv = os.path.join(OUT_DIR, f"slice_{slice_num}_classified_p85_otsu_realGT.csv")
        categorized_df.to_csv(out_csv, index=False)

        print(f"Saved: {out_png}")
        print(f"Saved: {out_fp}")
        print(f"Saved: {out_csv}")

    print("\nDone.")


if __name__ == "__main__":
    main()



--- Slice 110 ---
GT points=525 -> labels_unique=458 (oob=0, no_label=3)
bright_thr_intensity(p85.0)=78.000 | otsu_thr_pct_bright=41.992
Confusion: TP=303 FP=50 FN=155 TN=806
Saved: C:\Users\OSVALDO\Downloads\results\03fev\overlays_realGT_p85_otsu\slice_110_overlay_p85_otsu_realGT.png
Saved: C:\Users\OSVALDO\Downloads\results\03fev\overlays_realGT_p85_otsu\slice_110_FP_overlay_p85_otsu_realGT.tif
Saved: C:\Users\OSVALDO\Downloads\results\03fev\overlays_realGT_p85_otsu\slice_110_classified_p85_otsu_realGT.csv

--- Slice 300 ---
GT points=2142 -> labels_unique=648 (oob=0, no_label=6)
bright_thr_intensity(p85.0)=94.000 | otsu_thr_pct_bright=36.914
Confusion: TP=401 FP=368 FN=247 TN=1918
Saved: C:\Users\OSVALDO\Downloads\results\03fev\overlays_realGT_p85_otsu\slice_300_overlay_p85_otsu_realGT.png
Saved: C:\Users\OSVALDO\Downloads\results\03fev\overlays_realGT_p85_otsu\slice_300_FP_overlay_p85_otsu_realGT.tif
Saved: C:\Users\OSVALDO\Downloads\results\03fev\overlays_realGT_p85_otsu\slice_30